# SOLID Principle Detection — Tool Tester
Loads `python-codes.xlsx`, runs each detection tool against every row, and reports per-principle accuracy.

## 1 · Imports & tool loading

In [ ]:
import ast
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

from SOLID.SRP_Detection_Final           import get_srp_report
from SOLID.OCP_Detection_Final           import get_ocp_report
from SOLID.Liskov_Substitution_Principle import get_lsp_report
from SOLID.ISP_detect                    import get_isp_report
from SOLID.dependancy_principle          import get_dip_report
print('✅ All tools imported successfully.')


✅ All tools imported successfully.


## 2 · Load dataset

In [2]:
DATA_PATH = r'python-codes-clean.xlsx'

df = pd.read_excel(DATA_PATH, dtype=str)
df.columns = df.columns.str.strip().str.lower()

# df['code'] = (
#     df['code']
#     .str.strip("'\"")
#     .str.replace(r'\\n', '\n', regex=True)
#     .str.replace(r'\\t', '\t', regex=True)
# )

print(f'Loaded {len(df)} rows  |  columns: {list(df.columns)}')
display(df.head(3))

Loaded 641 rows  |  columns: ['id', 'language', 'code', 'srp', 'ocp', 'lsp', 'isp', 'dip']


,id,language,code,srp,ocp,lsp,isp,dip
0,1,Python,"class Writer:\n def __init__(self, type: in...",non-compliant,non-compliant,non-compliant,non-compliant,non-compliant
1,2,Python,class TelephoneDirectory:\n def __init__(self...,compliant,non-compliant,non-compliant,non-compliant,compliant
2,3,Python,import unittest\nclass Shape:\n def compute...,compliant,non-compliant,compliant,non-compliant,compliant


## 3 · Helper functions

In [3]:
def extract_status(report) -> str:
    """Normalise any report shape → 'Pass' | 'Violation' | 'Error'."""
    if isinstance(report, list):
        statuses = [r.get('status', 'Pass') for r in report]
        if any(s == 'Violation' for s in statuses):
            return 'Violation'
        return statuses[0] if statuses else 'Pass'
    if isinstance(report, dict):
        return report.get('status', 'Pass')
    return 'Pass'


def label_to_expected(label: str) -> str:
    """Map dataset label → tool status string."""
    label = str(label).strip().lower()
    if label in ('non-compliant', 'violation', 'fail'):
        return 'Violation'
    if label in ('compliant', 'pass', 'ok'):
        return 'Pass'
    return label.capitalize()


PRINCIPLES = [
    ('SRP', get_srp_report),
    ('OCP', get_ocp_report),
    ('LSP', get_lsp_report),
    ('ISP', get_isp_report),
    ('DIP', get_dip_report),
]

print('Helpers ready.')

Helpers ready.


## 4 · Run tests

In [4]:
records = []

for _, row in df.iterrows():
    code   = row.get('code', '')
    row_id = row.get('id', '?')

    for principle, tool_fn in PRINCIPLES:
        ground_truth = row.get(principle.lower(), 'compliant')
        expected     = label_to_expected(ground_truth)

        try:
            report = tool_fn(code)
            got    = extract_status(report)
            if got == 'Error':
                outcome = 'ERROR'
            elif got == expected:
                outcome = 'PASS'
            else:
                outcome = 'FAIL'
        except Exception as exc:
            got     = 'Error'
            outcome = 'ERROR'
            report  = str(exc)

        records.append({
            'id':        row_id,
            'code':      code,
            'principle': principle,
            'expected':  expected,
            'got':       got,
            'outcome':   outcome,
            'detail':    str(report) if outcome != 'PASS' else '',
        })

results_df = pd.DataFrame(records)
print(f'Tests complete: {len(results_df)} checks across {len(df)} rows.')
display(results_df.head(3))

<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:64: SyntaxWarning: invalid escape sequence '\/'
<unknown>:64: SyntaxWarning: invalid escape sequence '\/'


Syntax Error: unterminated string literal (detected at line 17) (<unknown>, line 17)
Syntax Error: unterminated string literal (detected at line 7) (<unknown>, line 7)


<unknown>:61: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'
<unknown>:61: SyntaxWarning: invalid escape sequence '\/'


Syntax Error: expected an indented block after function definition on line 2 (<unknown>, line 5)
Syntax Error: unterminated string literal (detected at line 26) (<unknown>, line 26)
Syntax Error: unmatched ']' (<unknown>, line 61)
Syntax Error: invalid syntax (<unknown>, line 31)
Syntax Error: unterminated string literal (detected at line 124) (<unknown>, line 124)
Tests complete: 3205 checks across 641 rows.


,id,code,principle,expected,got,outcome,detail
0,1,"class Writer:\n def __init__(self, type: in...",SRP,Violation,Pass,FAIL,"[{'class': 'Writer', 'status': 'Pass', 'reason..."
1,1,"class Writer:\n def __init__(self, type: in...",OCP,Violation,Pass,FAIL,"{'status': 'Pass', 'reason': 'No type-based di..."
2,1,"class Writer:\n def __init__(self, type: in...",LSP,Violation,Pass,FAIL,"{'status': 'Pass', 'reason': 'Subclasses maint..."


## 5 · Results — failures & errors

In [5]:
failures = results_df[results_df['outcome'] != 'PASS'].reset_index(drop=True)
print(f'Failures / Errors: {len(failures)}  |  Passed: {len(results_df) - len(failures)}')
display(failures[['id', 'code', 'principle', 'expected', 'got', 'outcome', 'detail']])

Failures / Errors: 1785  |  Passed: 1420


,id,code,principle,expected,got,outcome,detail
0,1,"class Writer:\n def __init__(self, type: in...",SRP,Violation,Pass,FAIL,"[{'class': 'Writer', 'status': 'Pass', 'reason..."
1,1,"class Writer:\n def __init__(self, type: in...",OCP,Violation,Pass,FAIL,"{'status': 'Pass', 'reason': 'No type-based di..."
2,1,"class Writer:\n def __init__(self, type: in...",LSP,Violation,Pass,FAIL,"{'status': 'Pass', 'reason': 'Subclasses maint..."
3,1,"class Writer:\n def __init__(self, type: in...",DIP,Violation,Pass,FAIL,"{'status': 'Pass', 'violations': [], 'reason':..."
4,2,class TelephoneDirectory:\n def __init__(self...,SRP,Pass,Violation,FAIL,"[{'class': 'TelephoneDirectory', 'status': 'Vi..."
...,...,...,...,...,...,...,...
1780,639,"from abc import ABC, abstractmethod\n\nclass O...",ISP,Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
1781,639,"from abc import ABC, abstractmethod\n\nclass O...",DIP,Pass,Violation,FAIL,"{'status': 'Violation', 'violations': [{'line'..."
1782,640,"class CalorieTracker:\n def __init__(self, ...",OCP,Violation,Pass,FAIL,"{'status': 'Pass', 'reason': 'No type-based di..."
1783,640,"class CalorieTracker:\n def __init__(self, ...",ISP,Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."


## 6 · Accuracy summary per principle

In [6]:
summary = (
    results_df
    .groupby('principle')
    .apply(
        lambda g: pd.Series({
            'Total':    len(g),
            'Correct':  (g['outcome'] == 'PASS').sum(),
            'Failures': (g['outcome'] == 'FAIL').sum(),
            'Errors':   (g['outcome'] == 'ERROR').sum(),
            'Accuracy': f"{(g['outcome'] == 'PASS').mean() * 100:.1f}%",
        }),
        include_groups=False,
    )
    .reindex(['SRP', 'OCP', 'LSP', 'ISP', 'DIP'])
)

overall_acc = (results_df['outcome'] == 'PASS').mean() * 100
display(summary)
print(f'\nOverall accuracy: {overall_acc:.1f}%  '
    f"({(results_df['outcome'] == 'PASS').sum()}/{len(results_df)})")

,Total,Correct,Failures,Errors,Accuracy
principle,,,,,
SRP,641,414,220,7,64.6%
OCP,641,132,509,0,20.6%
LSP,641,409,225,7,63.8%
ISP,641,97,544,0,15.1%
DIP,641,368,266,7,57.4%



Overall accuracy: 44.3%  (1420/3205)


## 7 · Per-principle deep-dive

In [7]:
# Change to any of: 'SRP', 'OCP', 'LSP', 'ISP', 'DIP'
FOCUS = 'ISP'

subset = results_df[results_df['principle'] == FOCUS].reset_index(drop=True)
display(subset[['id', 'code', 'expected', 'got', 'outcome', 'detail']])

,id,code,expected,got,outcome,detail
0,1,"class Writer:\n def __init__(self, type: in...",Violation,Violation,PASS,
1,2,class TelephoneDirectory:\n def __init__(self...,Violation,Violation,PASS,
2,3,import unittest\nclass Shape:\n def compute...,Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
3,4,"class StringEncrypter:\n def encrypt(self, ...",Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
4,5,import logging\nimport pandas as pd\nimport nu...,Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
...,...,...,...,...,...,...
636,637,"class Book:\n def __init__(self, title, aut...",Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
637,638,class Keyboard:\n def type(self):\n ...,Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
638,639,"from abc import ABC, abstractmethod\n\nclass O...",Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."
639,640,"class CalorieTracker:\n def __init__(self, ...",Violation,Pass,FAIL,"{'status': 'Pass', 'total': 0, 'high': 0, 'med..."


## 8 · Confusion matrix (per principle)

In [8]:
for principle, _ in PRINCIPLES:
    sub = results_df[results_df['principle'] == principle]
    cm  = pd.crosstab(
        sub['expected'], sub['got'],
        rownames=['Expected'], colnames=['Predicted'],
    )
    print(f'\n── {principle} ──────────────────────')
    display(cm)


── SRP ──────────────────────


Predicted,Error,Pass,Violation
Expected,,,
Pass,6,330,105
Violation,1,115,84



── OCP ──────────────────────


Predicted,Pass,Violation
Expected,,
Pass,123,0
Violation,509,9



── LSP ──────────────────────


Predicted,Error,Pass,Violation
Expected,,,
Pass,4,402,86
Violation,3,139,7



── ISP ──────────────────────


Predicted,Pass,Violation
Expected,,
Pass,15,1
Violation,543,82



── DIP ──────────────────────


Predicted,Error,Pass,Violation
Expected,,,
Pass,7,360,231
Violation,0,35,8


In [9]:
from sklearn.metrics import classification_report

for principle, _ in PRINCIPLES:

    sub = results_df[results_df['principle'] == principle]

    y_true = sub['expected']
    y_pred = sub['got']

    # Classification report as dict
    report_dict = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()

    print(f'\n── {principle} ──────────────────────')
    display(report_df)


── SRP ──────────────────────


,precision,recall,f1-score,support
Error,0.000000,0.000000,0.000000,0.000000
Pass,0.741573,0.748299,0.744921,441.000000
Violation,0.444444,0.420000,0.431877,200.000000
accuracy,0.645866,0.645866,0.645866,0.645866
macro avg,0.395339,0.389433,0.392266,641.000000
weighted avg,0.648865,0.645866,0.647247,641.000000



── OCP ──────────────────────


,precision,recall,f1-score,support
Pass,0.194620,1.000000,0.325828,123.000000
Violation,1.000000,0.017375,0.034156,518.000000
accuracy,0.205928,0.205928,0.205928,0.205928
macro avg,0.597310,0.508687,0.179992,641.000000
weighted avg,0.845458,0.205928,0.090124,641.000000



── LSP ──────────────────────


,precision,recall,f1-score,support
Error,0.000000,0.000000,0.000000,0.000000
Pass,0.743068,0.817073,0.778316,492.000000
Violation,0.075269,0.046980,0.057851,149.000000
accuracy,0.638066,0.638066,0.638066,0.638066
macro avg,0.272779,0.288018,0.278722,641.000000
weighted avg,0.587839,0.638066,0.610844,641.000000



── ISP ──────────────────────


,precision,recall,f1-score,support
Pass,0.026882,0.937500,0.052265,16.000000
Violation,0.987952,0.131200,0.231638,625.000000
accuracy,0.151326,0.151326,0.151326,0.151326
macro avg,0.507417,0.534350,0.141952,641.000000
weighted avg,0.963963,0.151326,0.227161,641.000000



── DIP ──────────────────────


,precision,recall,f1-score,support
Error,0.000000,0.000000,0.000000,0.000000
Pass,0.911392,0.602007,0.725076,598.000000
Violation,0.033473,0.186047,0.056738,43.000000
accuracy,0.574103,0.574103,0.574103,0.574103
macro avg,0.314955,0.262684,0.260604,641.000000
weighted avg,0.852499,0.574103,0.680242,641.000000


In [10]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

for principle, _ in PRINCIPLES:

    # Filter dataframe for current principle
    sub = results_df[results_df['principle'] == principle]

    # True labels and predictions
    y_true = sub['expected']
    y_pred = sub['got']

    # Confusion Matrix
    cm = pd.crosstab(
        y_true,
        y_pred,
        rownames=['Expected'],
        colnames=['Predicted']
    )

    print(f'\n── {principle} ──────────────────────')

    # Display confusion matrix
    print("\nConfusion Matrix:")
    display(cm)

    # Classification Report
    print("\nClassification Report:")
    report = classification_report(
        y_true,
        y_pred,
        digits=4,
        zero_division=0
    )

    print(report)


── SRP ──────────────────────

Confusion Matrix:


Predicted,Error,Pass,Violation
Expected,,,
Pass,6,330,105
Violation,1,115,84



Classification Report:
              precision    recall  f1-score   support

       Error     0.0000    0.0000    0.0000         0
        Pass     0.7416    0.7483    0.7449       441
   Violation     0.4444    0.4200    0.4319       200

    accuracy                         0.6459       641
   macro avg     0.3953    0.3894    0.3923       641
weighted avg     0.6489    0.6459    0.6472       641


── OCP ──────────────────────

Confusion Matrix:


Predicted,Pass,Violation
Expected,,
Pass,123,0
Violation,509,9



Classification Report:
              precision    recall  f1-score   support

        Pass     0.1946    1.0000    0.3258       123
   Violation     1.0000    0.0174    0.0342       518

    accuracy                         0.2059       641
   macro avg     0.5973    0.5087    0.1800       641
weighted avg     0.8455    0.2059    0.0901       641


── LSP ──────────────────────

Confusion Matrix:


...


Classification Report:
              precision    recall  f1-score   support

       Error     0.0000    0.0000    0.0000         0
        Pass     0.7431    0.8171    0.7783       492
   Violation     0.0753    0.0470    0.0579       149

    accuracy                         0.6381       641
   macro avg     0.2728    0.2880    0.2787       641
weighted avg     0.5878    0.6381    0.6108       641


── ISP ──────────────────────

Confusion Matrix:


Predicted,Pass,Violation
Expected,,
Pass,15,1
Violation,543,82



Classification Report:
              precision    recall  f1-score   support

        Pass     0.0269    0.9375    0.0523        16
   Violation     0.9880    0.1312    0.2316       625

    accuracy                         0.1513       641
   macro avg     0.5074    0.5343    0.1420       641
weighted avg     0.9640    0.1513    0.2272       641


── DIP ──────────────────────

Confusion Matrix:


Predicted,Error,Pass,Violation
Expected,,,
Pass,7,360,231
Violation,0,35,8



Classification Report:
              precision    recall  f1-score   support

       Error     0.0000    0.0000    0.0000         0
        Pass     0.9114    0.6020    0.7251       598
   Violation     0.0335    0.1860    0.0567        43

    accuracy                         0.5741       641
   macro avg     0.3150    0.2627    0.2606       641
weighted avg     0.8525    0.5741    0.6802       641

